# GDEX PREPBUFR NetCDF cleaning notebook

This notebook reads the GDEX subset output `prepbufr.844400.20240101.t00z.nc.gz`. This is no longer raw binary PREPBUFR: GDEX converted it with MET `pb2nc` into NetCDF. That means we can read latitude, longitude, station id, valid time, variable name, level, height, quality mark, and observation value directly.

Run from top to bottom. Change `NC_GZ_PATH` to inspect another cycle such as `t06z`, `t12z`, or `t18z`.


In [13]:
from __future__ import annotations

from pathlib import Path
import gzip
import shutil

import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import Markdown, display

BASE_DIR = Path.cwd()
NC_GZ_PATH = BASE_DIR / 'prepbufr.844400.20240101.t00z.nc.gz'
NC_PATH = NC_GZ_PATH.with_suffix('')  # removes .gz -> .nc

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 160)


## 1. Decompress and open the NetCDF file

The file from GDEX is `.nc.gz`. This cell creates a local `.nc` copy only if needed, then opens it with xarray.

In [14]:
assert NC_GZ_PATH.exists(), f'Missing file: {NC_GZ_PATH}'

if (not NC_PATH.exists()) or (NC_PATH.stat().st_mtime < NC_GZ_PATH.stat().st_mtime):
    with gzip.open(NC_GZ_PATH, 'rb') as src, NC_PATH.open('wb') as dst:
        shutil.copyfileobj(src, dst)

print(f'Compressed:   {NC_GZ_PATH.name} ({NC_GZ_PATH.stat().st_size / 1024**2:,.2f} MB)')
print(f'Decompressed: {NC_PATH.name} ({NC_PATH.stat().st_size / 1024**2:,.2f} MB)')

ds = xr.open_dataset(NC_PATH, engine='netcdf4')
ds


Compressed:   prepbufr.844400.20240101.t00z.nc.gz (8.19 MB)
Decompressed: prepbufr.844400.20240101.t00z.nc (63.85 MB)


<xarray.Dataset> Size: 93MB
Dimensions:        (nhdr: 310120, nobs: 2306273, npbhdr: 310120, nhdr_typ: 8,
                    nhdr_sid: 5651, nhdr_vld: 11526, nobs_qty: 5,
                    obs_var_num: 12)
Dimensions without coordinates: nhdr, nobs, npbhdr, nhdr_typ, nhdr_sid,
                                nhdr_vld, nobs_qty, obs_var_num
Data variables: (12/22)
    hdr_typ        (nhdr) float64 2MB ...
    hdr_sid        (nhdr) float64 2MB ...
    hdr_vld        (nhdr) float64 2MB ...
    hdr_lat        (nhdr) float32 1MB ...
    hdr_lon        (nhdr) float32 1MB ...
    hdr_elv        (nhdr) float32 1MB ...
    ...             ...
    hdr_sid_table  (nhdr_sid) |S40 226kB ...
    hdr_vld_table  (nhdr_vld) |S16 184kB ...
    obs_qty_table  (nobs_qty) |S16 80B ...
    obs_var        (obs_var_num) |S40 480B ...
    obs_unit       (obs_var_num) |S40 480B ...
    obs_desc       (obs_var_num) |S80 960B ...
Attributes:
    MET_Obs_version:  1.02
    use_var_id:       true
    FileOrigins:      File / generated 20260507_063408 UTC on host casper61 b...
    MET_version:      V12.2.0
    MET_tool:         pb2nc

## 2. What is inside?

This shows the dimensions, global metadata, and variables. The key split is: `hdr_*` variables describe report headers; `obs_*` variables describe individual observed values.

In [15]:
display(Markdown('### Dimensions'))
display(pd.DataFrame({'dimension': list(ds.sizes.keys()), 'size': list(ds.sizes.values())}))

display(Markdown('### Global attributes'))
display(pd.DataFrame([{'attribute': k, 'value': v} for k, v in ds.attrs.items()]))

var_rows = []
for name, var in ds.variables.items():
    var_rows.append({
        'name': name,
        'dims': ', '.join(var.dims),
        'shape': var.shape,
        'dtype': str(var.dtype),
        'long_name': var.attrs.get('long_name'),
        'units': var.attrs.get('units'),
    })
variables = pd.DataFrame(var_rows)
display(variables)


### Dimensions

,dimension,size
0,nhdr,310120
1,nobs,2306273
2,npbhdr,310120
3,nhdr_typ,8
4,nhdr_sid,5651
5,nhdr_vld,11526
6,nobs_qty,5
7,obs_var_num,12


### Global attributes

,attribute,value
0,MET_Obs_version,1.02
1,use_var_id,true
2,FileOrigins,File / generated 20260507_063408 UTC on host casper61 by the MET pb2nc tool
3,MET_version,V12.2.0
4,MET_tool,pb2nc


,name,dims,shape,dtype,long_name,units
0,hdr_typ,nhdr,"(310120,)",float64,index of message type,None
1,hdr_sid,nhdr,"(310120,)",float64,index of station identification,None
2,hdr_vld,nhdr,"(310120,)",float64,index of valid time,None
3,hdr_lat,nhdr,"(310120,)",float32,latitude,degrees_north
4,hdr_lon,nhdr,"(310120,)",float32,longitude,degrees_east
5,hdr_elv,nhdr,"(310120,)",float32,elevation,meters above sea level (msl)
6,obs_qty,nobs,"(2306273,)",int32,index of quality flag,None
7,obs_hid,nobs,"(2306273,)",float64,index of matching header data,None
8,obs_vid,nobs,"(2306273,)",float64,index of BUFR variable corresponding to the observation type,None
9,obs_lvl,nobs,"(2306273,)",float32,pressure level (hPa) or accumulation interval (sec),None


## 3. Helper functions

These decode the lookup tables and build readable pandas tables. The NetCDF stores station IDs, times, observation types, quality marks, and variable names as indexed lookup tables.

In [16]:
def decode_bytes(value) -> str:
    if isinstance(value, (bytes, bytearray, np.bytes_)):
        return value.decode('utf-8', errors='ignore').strip('\x00 ').strip()
    return str(value).strip()


def decode_table(name: str) -> pd.Series:
    return pd.Series([decode_bytes(v) for v in ds[name].values], name=name)


def indexed_lookup(index_values, table: pd.Series) -> np.ndarray:
    idx = np.asarray(index_values).astype(int)
    return table.iloc[idx].to_numpy()


def build_header_table() -> pd.DataFrame:
    type_table = decode_table('hdr_typ_table')
    sid_table = decode_table('hdr_sid_table')
    vld_table = decode_table('hdr_vld_table')

    raw_valid = indexed_lookup(ds['hdr_vld'].values, vld_table)
    valid_time = pd.to_datetime(raw_valid, format='%Y%m%d_%H%M%S', errors='coerce', utc=True)

    return pd.DataFrame({
        'header_id': np.arange(ds.sizes['nhdr']),
        'obs_type': indexed_lookup(ds['hdr_typ'].values, type_table),
        'station_id': indexed_lookup(ds['hdr_sid'].values, sid_table),
        'valid_time': valid_time,
        'raw_valid_time': raw_valid,
        'lat': ds['hdr_lat'].values,
        'lon': ds['hdr_lon'].values,
        'elevation_m': ds['hdr_elv'].values,
        'prepbufr_report_type': ds['hdr_prpt_typ'].values,
        'input_report_type': ds['hdr_irpt_typ'].values,
        'instrument_type': ds['hdr_inst_typ'].values,
    })


def build_observation_table(start: int = 0, stop: int | None = 100_000, merge_headers: bool = True) -> pd.DataFrame:
    if stop is None:
        stop = ds.sizes['nobs']
    if start < 0 or stop < start:
        raise ValueError('Use 0 <= start <= stop')

    obs_slice = slice(start, stop)
    var_table = decode_table('obs_var')
    unit_table = decode_table('obs_unit')
    desc_table = decode_table('obs_desc')
    qty_table = decode_table('obs_qty_table')

    vid = ds['obs_vid'].values[obs_slice].astype(int)
    qty_idx = ds['obs_qty'].values[obs_slice].astype(int)

    obs = pd.DataFrame({
        'obs_id': np.arange(start, stop),
        'header_id': ds['obs_hid'].values[obs_slice].astype(int),
        'var_name': var_table.iloc[vid].to_numpy(),
        'var_unit': unit_table.iloc[vid].to_numpy(),
        'var_description': desc_table.iloc[vid].to_numpy(),
        'quality_mark': qty_table.iloc[qty_idx].to_numpy(),
        'quality_index': qty_idx,
        'pressure_level_hpa_or_accum_sec': ds['obs_lvl'].values[obs_slice],
        'height_m': ds['obs_hgt'].values[obs_slice],
        'value': ds['obs_val'].values[obs_slice],
    })

    if merge_headers:
        obs = obs.merge(headers, on='header_id', how='left')
    return obs


def filter_observations(
    obs: pd.DataFrame,
    obs_types: list[str] | None = None,
    var_names: list[str] | None = None,
    quality_marks: list[str] | None = None,
    lat_range: tuple[float, float] | None = None,
    lon_range: tuple[float, float] | None = None,
) -> pd.DataFrame:
    out = obs
    if obs_types:
        out = out[out['obs_type'].isin(obs_types)]
    if var_names:
        out = out[out['var_name'].isin(var_names)]
    if quality_marks:
        out = out[out['quality_mark'].isin([str(q) for q in quality_marks])]
    if lat_range:
        out = out[out['lat'].between(lat_range[0], lat_range[1])]
    if lon_range:
        out = out[out['lon'].between(lon_range[0], lon_range[1])]
    return out.copy()


def make_wide_observation_table(obs: pd.DataFrame) -> pd.DataFrame:
    index_cols = [
        'header_id', 'station_id', 'valid_time', 'lat', 'lon', 'elevation_m', 'obs_type',
        'prepbufr_report_type', 'input_report_type', 'instrument_type',
        'pressure_level_hpa_or_accum_sec', 'height_m',
    ]
    return (
        obs.pivot_table(index=index_cols, columns='var_name', values='value', aggfunc='first')
        .reset_index()
        .rename_axis(columns=None)
    )


## 4. Lookup tables

These explain coded values in the NetCDF file.

In [17]:
lookup_tables = {
    'observation_types': decode_table('hdr_typ_table'),
    'quality_marks': decode_table('obs_qty_table'),
    'variables': pd.DataFrame({
        'var_name': decode_table('obs_var'),
        'unit': decode_table('obs_unit'),
        'description': decode_table('obs_desc'),
    }),
}

print('Observation types:')
display(lookup_tables['observation_types'].to_frame('obs_type'))

print('Quality marks present in lookup table:')
display(lookup_tables['quality_marks'].to_frame('quality_mark'))

print('Variables:')
display(lookup_tables['variables'])


Observation types:


,obs_type
0,ADPUPA
1,AIRCAR
2,AIRCFT
3,VADWND
4,ADPSFC
5,SFCSHP
6,RASSDA
7,ASCATW


Quality marks present in lookup table:


,quality_mark
0,2
1,9
2,3
3,8
4,1


Variables:


,var_name,unit,description
0,PRES,PASCALS,PRESSURE OBSERVATION
1,SPFH,KG/KG,SPECIFIC HUMIDITY OBSERVATION
2,TMP,KELVIN,TEMPERATURE OBSERVATION
3,HGT,METER,HEIGHT OBSERVATION
4,UGRD,M/S,U-COMPONENT WIND OBSERVATION
5,VGRD,M/S,V-COMPONENT WIND OBSERVATION
6,DPT,KELVIN,DERIVED DEWPOINT OBSERVATION
7,WDIR,DEGREES,DERIVED WIND DIRECTION OBSERVATION
8,WIND,M/S,DERIVED WIND SPEED OBSERVATION
9,RH,PERCENT,DERIVED RELATIVE HUMIDITY OBSERVATION


## 5. Header table: one row per report header

A header is the shared metadata for one report/profile/platform/time/location. This is where latitude, longitude, station ID, and valid time live.

In [18]:
headers = build_header_table()
print(f'Header rows: {len(headers):,}')
display(headers.head(20))

display(Markdown('### Header counts by observation type'))
display(headers.groupby('obs_type').size().reset_index(name='header_count').sort_values('header_count', ascending=False))

display(Markdown('### Spatial extent'))
display(headers[['lat', 'lon', 'elevation_m']].describe())


Header rows: 310,120


,header_id,obs_type,station_id,valid_time,raw_valid_time,lat,lon,elevation_m,prepbufr_report_type,input_report_type,instrument_type
0,0,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.480000,-81.699997,10.0,120.0,11.0,154.0
1,1,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.480000,-81.699997,10.0,220.0,11.0,154.0
2,2,ADPUPA,72248,2024-01-01 00:00:00+00:00,20240101_000000,32.450001,-93.839996,85.0,120.0,11.0,154.0
3,3,ADPUPA,72248,2024-01-01 00:00:00+00:00,20240101_000000,32.450001,-93.839996,85.0,220.0,11.0,154.0
4,4,ADPUPA,72249,2024-01-01 00:00:00+00:00,20240101_000000,32.840000,-97.300003,195.0,120.0,11.0,141.0
5,5,ADPUPA,72249,2024-01-01 00:00:00+00:00,20240101_000000,32.840000,-97.300003,195.0,220.0,11.0,141.0
6,6,ADPUPA,72393,2024-01-01 00:00:00+00:00,20240101_000000,34.750000,-120.570000,100.0,120.0,11.0,141.0
7,7,ADPUPA,72393,2024-01-01 00:00:00+00:00,20240101_000000,34.750000,-120.570000,100.0,220.0,11.0,141.0
8,8,ADPUPA,72305,2024-01-01 00:00:00+00:00,20240101_000000,34.779999,-76.879997,11.0,120.0,11.0,154.0
9,9,ADPUPA,72305,2024-01-01 00:00:00+00:00,20240101_000000,34.779999,-76.879997,11.0,220.0,11.0,154.0


### Header counts by observation type

,obs_type,header_count
2,AIRCAR,235194
0,ADPSFC,49760
6,SFCSHP,22074
7,VADWND,1734
3,AIRCFT,1065
5,RASSDA,160
1,ADPUPA,132
4,ASCATW,1


### Spatial extent

,lat,lon,elevation_m
count,310120.000000,310120.000000,310120.000000
mean,36.775990,-95.908356,4480.167480
std,5.225873,15.014267,4224.176758
min,24.379999,-124.636940,-104.000000
25%,33.051998,-108.883003,514.000000
50%,36.779999,-94.434998,3135.000000
75%,40.560001,-82.946999,8768.000000
max,49.790501,-66.995003,13120.000000


## 6. Observation table: one row per observed value

This is the long/tidy table. One row is one variable value attached to a header. Example: one `TMP` temperature value at a station/time/level.

In [19]:
# Start with a sample so the notebook stays responsive. Set OBS_STOP = None to load all observations.
OBS_START = 0
OBS_STOP = 200_000

obs_sample = build_observation_table(OBS_START, OBS_STOP, merge_headers=True)
print(f'Observation rows loaded: {len(obs_sample):,} of {ds.sizes["nobs"]:,}')
display(obs_sample.head(30))


Observation rows loaded: 200,000 of 2,306,273


,obs_id,header_id,var_name,var_unit,var_description,quality_mark,quality_index,pressure_level_hpa_or_accum_sec,height_m,value,obs_type,station_id,valid_time,raw_valid_time,lat,lon,elevation_m,prepbufr_report_type,input_report_type,instrument_type
0,0,0,PRES,PASCALS,PRESSURE OBSERVATION,2,0,1021.0,9.992178,102100.000000,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.48,-81.699997,10.0,120.0,11.0,154.0
1,1,0,SPFH,KG/KG,SPECIFIC HUMIDITY OBSERVATION,2,0,1021.0,9.992178,0.006000,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.48,-81.699997,10.0,120.0,11.0,154.0
2,2,0,TMP,KELVIN,TEMPERATURE OBSERVATION,2,0,1021.0,9.992178,282.950012,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.48,-81.699997,10.0,120.0,11.0,154.0
3,3,0,HGT,METER,HEIGHT OBSERVATION,2,0,1021.0,9.992178,10.000000,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.48,-81.699997,10.0,120.0,11.0,154.0
4,4,0,DPT,KELVIN,DERIVED DEWPOINT OBSERVATION,2,0,1021.0,9.992178,279.835327,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.48,-81.699997,10.0,120.0,11.0,154.0
5,5,0,RH,PERCENT,DERIVED RELATIVE HUMIDITY OBSERVATION,2,0,1021.0,9.992178,81.347183,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.48,-81.699997,10.0,120.0,11.0,154.0
6,6,0,MIXR,KG/KG,DERIVED MIXING RATIO OBSERVATION,2,0,1021.0,9.992178,0.006036,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.48,-81.699997,10.0,120.0,11.0,154.0
7,7,0,PRES,PASCALS,PRESSURE OBSERVATION,2,0,1010.0,NaN,101000.000000,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.48,-81.699997,10.0,120.0,11.0,154.0
8,8,0,SPFH,KG/KG,SPECIFIC HUMIDITY OBSERVATION,2,0,1010.0,NaN,0.004747,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.48,-81.699997,10.0,120.0,11.0,154.0
9,9,0,TMP,KELVIN,TEMPERATURE OBSERVATION,2,0,1010.0,NaN,287.350006,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.48,-81.699997,10.0,120.0,11.0,154.0


## 7. Summaries of the loaded observation sample

Use these tables to check what variables and observation types dominate the file.

In [20]:
display(Markdown('### Rows by observation type'))
display(obs_sample.groupby('obs_type').size().reset_index(name='rows').sort_values('rows', ascending=False))

display(Markdown('### Rows by variable'))
display(obs_sample.groupby(['var_name', 'var_unit', 'var_description']).size().reset_index(name='rows').sort_values('rows', ascending=False))

display(Markdown('### Rows by observation type and variable'))
display(obs_sample.groupby(['obs_type', 'var_name']).size().reset_index(name='rows').sort_values(['obs_type', 'rows'], ascending=[True, False]).head(100))

display(Markdown('### Quality mark counts'))
display(obs_sample.groupby('quality_mark').size().reset_index(name='rows').sort_values('rows', ascending=False))


### Rows by observation type

,obs_type,rows
1,AIRCAR,111175
0,ADPUPA,88825


### Rows by variable

,var_name,var_unit,var_description,rows
3,PRES,PASCALS,PRESSURE OBSERVATION,39914
1,HGT,METER,HEIGHT OBSERVATION,28536
7,UGRD,M/S,U-COMPONENT WIND OBSERVATION,21527
8,VGRD,M/S,V-COMPONENT WIND OBSERVATION,21527
10,WIND,M/S,DERIVED WIND SPEED OBSERVATION,21527
9,WDIR,DEGREES,DERIVED WIND DIRECTION OBSERVATION,21483
6,TMP,KELVIN,TEMPERATURE OBSERVATION,17584
5,SPFH,KG/KG,SPECIFIC HUMIDITY OBSERVATION,7024
2,MIXR,KG/KG,DERIVED MIXING RATIO OBSERVATION,7023
0,DPT,KELVIN,DERIVED DEWPOINT OBSERVATION,6937


### Rows by observation type and variable

,obs_type,var_name,rows
3,ADPUPA,PRES,15747
7,ADPUPA,UGRD,9914
8,ADPUPA,VGRD,9914
10,ADPUPA,WIND,9914
9,ADPUPA,WDIR,9870
6,ADPUPA,TMP,5952
2,ADPUPA,MIXR,5829
5,ADPUPA,SPFH,5829
0,ADPUPA,DPT,5743
4,ADPUPA,RH,5743


### Quality mark counts

,quality_mark,rows
1,2,124139
0,1,60015
4,9,15775
2,3,61
3,8,10


## 8. Filter to the observation/variable slice you want

Change these lists to create a smaller table. This is the part you will eventually turn into model input.

In [25]:
# Examples:
#   OBS_TYPES = ['ADPSFC'] for surface land only
#   OBS_TYPES = ['ADPUPA'] for radiosonde/upper-air only
#   VAR_NAMES = ['TMP', 'UGRD', 'VGRD', 'SPFH', 'PRES'] for core state variables
OBS_TYPES = ['ADPSFC', 'ADPUPA', 'SFCSHP']
VAR_NAMES = ['TMP', 'UGRD', 'VGRD', 'SPFH', 'PRES', 'HGT']
QUALITY_MARKS = None  # e.g. ['1', '2', '3', '8', '9']; leave None while exploring
LAT_RANGE = None      # e.g. (20, 55)
LON_RANGE = None      # e.g. (-130, -60)

filtered = filter_observations(
    obs_sample,
    obs_types=OBS_TYPES,
    var_names=VAR_NAMES,
    quality_marks=QUALITY_MARKS,
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
)

print(f'Filtered rows: {len(filtered):,}')
display(filtered.head(1000))

display(filtered.groupby(['obs_type', 'var_name']).size().reset_index(name='rows').sort_values(['obs_type', 'var_name']))


Filtered rows: 51,726


,obs_id,header_id,var_name,var_unit,var_description,quality_mark,quality_index,pressure_level_hpa_or_accum_sec,height_m,value,obs_type,station_id,valid_time,raw_valid_time,lat,lon,elevation_m,prepbufr_report_type,input_report_type,instrument_type
0,0,0,PRES,PASCALS,PRESSURE OBSERVATION,2,0,1021.000000,9.992178,102100.000000,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.480000,-81.699997,10.0,120.0,11.0,154.0
1,1,0,SPFH,KG/KG,SPECIFIC HUMIDITY OBSERVATION,2,0,1021.000000,9.992178,0.006000,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.480000,-81.699997,10.0,120.0,11.0,154.0
2,2,0,TMP,KELVIN,TEMPERATURE OBSERVATION,2,0,1021.000000,9.992178,282.950012,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.480000,-81.699997,10.0,120.0,11.0,154.0
3,3,0,HGT,METER,HEIGHT OBSERVATION,2,0,1021.000000,9.992178,10.000000,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.480000,-81.699997,10.0,120.0,11.0,154.0
7,7,0,PRES,PASCALS,PRESSURE OBSERVATION,2,0,1010.000000,NaN,101000.000000,ADPUPA,72206,2024-01-01 00:00:00+00:00,20240101_000000,30.480000,-81.699997,10.0,120.0,11.0,154.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1764,1764,2,PRES,PASCALS,PRESSURE OBSERVATION,2,0,74.800003,NaN,7480.000488,ADPUPA,72248,2024-01-01 00:00:00+00:00,20240101_000000,32.450001,-93.839996,85.0,120.0,11.0,154.0
1765,1765,2,SPFH,KG/KG,SPECIFIC HUMIDITY OBSERVATION,9,1,74.800003,NaN,0.000001,ADPUPA,72248,2024-01-01 00:00:00+00:00,20240101_000000,32.450001,-93.839996,85.0,120.0,11.0,154.0
1766,1766,2,TMP,KELVIN,TEMPERATURE OBSERVATION,2,0,74.800003,NaN,211.050003,ADPUPA,72248,2024-01-01 00:00:00+00:00,20240101_000000,32.450001,-93.839996,85.0,120.0,11.0,154.0
1770,1770,2,PRES,PASCALS,PRESSURE OBSERVATION,2,0,74.300003,NaN,7430.000488,ADPUPA,72248,2024-01-01 00:00:00+00:00,20240101_000000,32.450001,-93.839996,85.0,120.0,11.0,154.0


,obs_type,var_name,rows
0,ADPUPA,HGT,4370
1,ADPUPA,PRES,15747
2,ADPUPA,SPFH,5829
3,ADPUPA,TMP,5952
4,ADPUPA,UGRD,9914
5,ADPUPA,VGRD,9914


## 9. Wide ML-friendly table

This pivots the long observation table so variables become columns. Rows are unique combinations of station/time/location/type/level/height. This is closer to what you feed into model preprocessing.

In [22]:
wide = make_wide_observation_table(filtered)
print(f'Wide rows: {len(wide):,}')
display(wide.head(30))

display(Markdown('### Missingness by column'))
missing = wide.isna().mean().sort_values(ascending=False).reset_index()
missing.columns = ['column', 'missing_fraction']
display(missing)


Wide rows: 4,386


,header_id,station_id,valid_time,lat,lon,elevation_m,obs_type,prepbufr_report_type,input_report_type,instrument_type,pressure_level_hpa_or_accum_sec,height_m,HGT,PRES,SPFH,TMP,UGRD,VGRD
0,0,72206,2024-01-01 00:00:00+00:00,30.48,-81.699997,10.0,ADPUPA,120.0,11.0,154.0,8.000000,31978.964844,32004.0,800.0,0.000033,218.449997,NaN,NaN
1,0,72206,2024-01-01 00:00:00+00:00,30.48,-81.699997,10.0,ADPUPA,120.0,11.0,154.0,10.000000,30586.056641,30610.0,1000.0,0.000027,218.649994,NaN,NaN
2,0,72206,2024-01-01 00:00:00+00:00,30.48,-81.699997,10.0,ADPUPA,120.0,11.0,154.0,20.000000,26219.474609,26240.0,2000.0,0.000007,213.649994,NaN,NaN
3,0,72206,2024-01-01 00:00:00+00:00,30.48,-81.699997,10.0,ADPUPA,120.0,11.0,154.0,30.000000,23701.445312,23720.0,3000.0,0.000004,211.449997,NaN,NaN
4,0,72206,2024-01-01 00:00:00+00:00,30.48,-81.699997,10.0,ADPUPA,120.0,11.0,154.0,50.000000,20563.902344,20580.0,5000.0,0.000002,209.050003,NaN,NaN
5,0,72206,2024-01-01 00:00:00+00:00,30.48,-81.699997,10.0,ADPUPA,120.0,11.0,154.0,70.000000,18495.521484,18510.0,7000.0,0.000001,209.850006,NaN,NaN
6,0,72206,2024-01-01 00:00:00+00:00,30.48,-81.699997,10.0,ADPUPA,120.0,11.0,154.0,100.000000,16317.226562,16330.0,10000.0,0.000001,207.849991,NaN,NaN
7,0,72206,2024-01-01 00:00:00+00:00,30.48,-81.699997,10.0,ADPUPA,120.0,11.0,154.0,150.000000,13819.181641,13830.0,15000.0,0.000002,214.449997,NaN,NaN
8,0,72206,2024-01-01 00:00:00+00:00,30.48,-81.699997,10.0,ADPUPA,120.0,11.0,154.0,200.000000,12000.605469,12010.0,20000.0,0.000008,218.850006,NaN,NaN
9,0,72206,2024-01-01 00:00:00+00:00,30.48,-81.699997,10.0,ADPUPA,120.0,11.0,154.0,250.000000,10571.723633,10580.0,25000.0,0.000032,222.850006,NaN,NaN


### Missingness by column

,column,missing_fraction
0,SPFH,0.795714
1,TMP,0.786822
2,VGRD,0.225946
3,UGRD,0.225946
4,PRES,0.005016
5,HGT,0.003648
6,station_id,0.000000
7,height_m,0.000000
8,pressure_level_hpa_or_accum_sec,0.000000
9,header_id,0.000000


## 10. Optional: open all four Jan 1 cycles

Use this after you understand one cycle. It reads all `.nc.gz` files in this folder and builds one combined header summary.

In [23]:
def open_gz_netcdf(path: Path) -> xr.Dataset:
    nc_path = path.with_suffix('')
    if (not nc_path.exists()) or (nc_path.stat().st_mtime < path.stat().st_mtime):
        with gzip.open(path, 'rb') as src, nc_path.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    return xr.open_dataset(nc_path, engine='netcdf4')

cycle_files = sorted(BASE_DIR.glob('prepbufr.844400.20240101.t*z.nc.gz'))
print([p.name for p in cycle_files])

cycle_summary_rows = []
for p in cycle_files:
    dsi = open_gz_netcdf(p)
    cycle_summary_rows.append({
        'file': p.name,
        'nhdr': dsi.sizes['nhdr'],
        'nobs': dsi.sizes['nobs'],
        'obs_types': ', '.join(decode_bytes(v) for v in dsi['hdr_typ_table'].values),
    })

cycle_summary = pd.DataFrame(cycle_summary_rows)
display(cycle_summary)


['prepbufr.844400.20240101.t00z.nc.gz', 'prepbufr.844400.20240101.t06z.nc.gz', 'prepbufr.844400.20240101.t12z.nc.gz', 'prepbufr.844400.20240101.t18z.nc.gz']


,file,nhdr,nobs,obs_types
0,prepbufr.844400.20240101.t00z.nc.gz,310120,2306273,"ADPUPA, AIRCAR, AIRCFT, VADWND, ADPSFC, SFCSHP, RASSDA, ASCATW"
1,prepbufr.844400.20240101.t06z.nc.gz,163764,1418274,"ADPUPA, AIRCAR, AIRCFT, VADWND, ADPSFC, SFCSHP, RASSDA"
2,prepbufr.844400.20240101.t12z.nc.gz,178807,1427638,"ADPUPA, AIRCAR, AIRCFT, VADWND, ADPSFC, SFCSHP, RASSDA"
3,prepbufr.844400.20240101.t18z.nc.gz,308355,1997754,"AIRCAR, AIRCFT, VADWND, ADPSFC, SFCSHP, RASSDA"


## Reading this dataset correctly

`headers` tells you **where and when** the report happened: station/platform ID, valid time, latitude, longitude, elevation, and report type.

`obs_sample` tells you **what was observed**: variable name, units, pressure/height, quality mark, and value.

The join key is `header_id`. Every observation row points back to one header row.